<a href="https://colab.research.google.com/github/AkhilanairK-healthcare/akhila-hospital-readmission-quality/blob/main/notebooks/sql_queries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import sqlite3
BASE = '/content/drive/MyDrive/Akhila-portfolio/Project2-data/'
conn = sqlite3.connect(BASE + 'readmissions.db')
def q(sql):
    return pd.read_sql_query(sql, conn)
print('connected')

connected


In [3]:
q("""SELECT condition, COUNT(excess_readmission_ratio) AS hospitals_with_data,
  ROUND(AVG(excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions GROUP BY condition ORDER BY avg_err DESC;""")
q("""SELECT condition, COUNT(excess_readmission_ratio) AS hospitals_with_data,
  ROUND(100.0 * AVG(above_benchmark), 1) AS pct_above_benchmark
FROM fact_readmissions GROUP BY condition ORDER BY pct_above_benchmark DESC;""")

,condition,hospitals_with_data,pct_above_benchmark
0,CABG (Bypass Surgery),878,49.9
1,Heart Attack (AMI),1736,49.7
2,Heart Failure,2621,48.9
3,Hip/Knee Replacement,1447,47.9
4,COPD,2323,47.2
5,Pneumonia,2715,46.8


In [4]:
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'Heart Failure' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'Pneumonia' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'Heart Attack (AMI)' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'COPD' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'Hip/Knee Replacement' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")
q("""SELECT facility_name, state, number_of_discharges,
  excess_readmission_ratio, number_of_readmissions
FROM fact_readmissions
WHERE condition = 'CABG (Bypass Surgery)' AND excess_readmission_ratio IS NOT NULL
ORDER BY excess_readmission_ratio DESC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio,number_of_readmissions
0,ROBERT WOOD JOHNSON UNIVERSITY HOSPITAL,NJ,357.0,1.4237,68.0
1,MCLAREN BAY REGION,MI,103.0,1.3880,25.0
2,PHYSICIANS REGIONAL MEDICAL CENTER,TN,149.0,1.3264,28.0
3,JOHNSON CITY MEDICAL CENTER,TN,101.0,1.2914,19.0
4,UNIVERSITY OF ALABAMA HOSPITAL,AL,358.0,1.2863,61.0
5,CHARLESTON AREA MEDICAL CENTER,WV,298.0,1.2837,48.0
6,MERCY HOSPITAL ST LOUIS,MO,175.0,1.2812,32.0
7,HCA FLORIDA BAYONET POINT HOSPITAL,FL,85.0,1.2553,20.0
8,DOCTORS MEDICAL CENTER,CA,130.0,1.2466,22.0
9,BAYSTATE MEDICAL CENTER,MA,310.0,1.2464,49.0


In [5]:
q("""SELECT state, COUNT(excess_readmission_ratio) AS hospital_measures,
  ROUND(AVG(excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions GROUP BY state
HAVING hospital_measures >= 100 ORDER BY avg_err DESC LIMIT 15;""")
q("""SELECT h.hospital_ownership, COUNT(*) AS hospital_measures,
  ROUND(AVG(f.excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions f JOIN dim_hospital h ON f.facility_id = h.facility_id
GROUP BY h.hospital_ownership ORDER BY avg_err DESC;""")
q("""SELECT h.hospital_type, COUNT(*) AS hospital_measures,
  ROUND(AVG(f.excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions f JOIN dim_hospital h ON f.facility_id = h.facility_id
GROUP BY h.hospital_type ORDER BY avg_err DESC;""")
q("""SELECT h.hospital_overall_rating, COUNT(*) AS hospital_measures,
  ROUND(AVG(f.excess_readmission_ratio), 4) AS avg_err
FROM fact_readmissions f JOIN dim_hospital h ON f.facility_id = h.facility_id
WHERE h.hospital_overall_rating IS NOT NULL
GROUP BY h.hospital_overall_rating ORDER BY h.hospital_overall_rating;""")

,hospital_overall_rating,hospital_measures,avg_err
0,1.0,1020,1.0484
1,2.0,3402,1.0250
2,3.0,5052,1.0061
3,4.0,4656,0.9855
4,5.0,1740,0.9663


In [6]:
q("""SELECT COUNT(DISTINCT facility_id) AS hospitals_in_hrrp,
  COUNT(*) AS hospital_measure_rows,
  COUNT(excess_readmission_ratio) AS rows_with_err
FROM fact_readmissions;""")
q("""SELECT condition, COUNT(*) AS rows_total,
  SUM(CASE WHEN excess_readmission_ratio IS NULL THEN 1 ELSE 0 END) AS rows_missing_err,
  ROUND(100.0 * SUM(CASE WHEN excess_readmission_ratio IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing
FROM fact_readmissions GROUP BY condition ORDER BY pct_missing DESC;""")
q("""SELECT facility_name, state, number_of_discharges, excess_readmission_ratio
FROM fact_readmissions
WHERE condition = 'Pneumonia' AND excess_readmission_ratio IS NOT NULL
  AND number_of_discharges >= 100
ORDER BY excess_readmission_ratio ASC LIMIT 15;""")

,facility_name,state,number_of_discharges,excess_readmission_ratio
0,AUGUSTA HEALTH,VA,392.0,0.7967
1,TRINITY HEALTH MUSKEGON HOSPITAL,MI,330.0,0.7976
2,KOOTENAI HEALTH,ID,413.0,0.8077
3,SALINA REGIONAL HEALTH CENTER,KS,317.0,0.8166
4,CHRISTUS ST VINCENT REGIONAL MEDICAL CENTER,NM,480.0,0.8174
5,HOLMES REGIONAL MEDICAL CENTER,FL,710.0,0.8175
6,MERCY MEDICAL CENTER,OR,216.0,0.8178
7,NORTHWELL HOSPITAL GLEN COVE,NY,234.0,0.8207
8,LOGAN HEALTH MEDICAL CENTER,MT,252.0,0.8245
9,PROVIDENCE ST JOSEPH HOSPITAL,CA,431.0,0.8248


In [4]:
q("""
SELECT facility_name, state, excess_readmission_ratio, state_rank
FROM (
SELECT
facility_name,
state,
excess_readmission_ratio,
RANK() OVER (PARTITION BY state ORDER BY excess_readmission_ratio DESC) AS state_rank
FROM fact_readmissions
WHERE condition = 'Heart Failure'
AND excess_readmission_ratio IS NOT NULL
)
WHERE state_rank <= 3
ORDER BY state, state_rank;
""")

,facility_name,state,excess_readmission_ratio,state_rank
0,CENTRAL PENINSULA GENERAL HOSPITAL,AK,1.2271,1
1,ALASKA REGIONAL HOSPITAL,AK,1.0988,2
2,ALASKA NATIVE MEDICAL CENTER,AK,1.0516,3
3,RIVERVIEW REGIONAL MEDICAL CENTER,AL,1.1060,1
4,BAPTIST MEDICAL CENTER EAST,AL,1.1038,2
...,...,...,...,...
148,RALEIGH GENERAL HOSPITAL,WV,1.1072,2
149,MON HEALTH MEDICAL CENTER,WV,1.0804,3
150,SAGEWEST HEALTH CARE,WY,0.9964,1
151,SHERIDAN MEMORIAL HOSPITAL,WY,0.9901,2
